# nuScenes — Xem point cloud LiDAR 3D + bounding box trên Google Colab

Notebook này hiển thị **point cloud 3D tương tác** (xoay, zoom bằng chuột) kèm **bounding box 3D** của nuScenes, dùng **Plotly** vì Open3D không mở được cửa sổ trên Colab.

**Thứ tự chạy:** Cell 1 → 2 → 3 → 4 → 5 → 6. Các cell 7, 8 là tuỳ chọn.

**Chuẩn bị trước:** tải `v1.0-mini.tgz` (~4 GB) từ https://www.nuscenes.org/download rồi **upload lên Google Drive** của bạn.

## Cell 1 — Cài thư viện

Nếu Colab báo *"RESTART SESSION"*, hãy bấm nút đó rồi chạy tiếp từ Cell 2 (không cần chạy lại Cell 1).

In [ ]:
!pip install -q nuscenes-devkit plotly

import numpy, plotly, matplotlib
print('numpy     ', numpy.__version__)
print('plotly    ', plotly.__version__)
print('matplotlib', matplotlib.__version__)
import nuscenes
print('nuscenes-devkit OK')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 42.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 114.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 13.5 MB/s eta 0:00:00


## Cell 2 — Kết nối Google Drive và giải nén dữ liệu

Sửa `TGZ_PATH` cho đúng vị trí file `v1.0-mini.tgz` trong Drive của bạn.

Giải nén ra `/content` (ổ đĩa cục bộ của Colab) nhanh hơn nhiều so với đọc thẳng từ Drive. Mất khoảng 3–6 phút, chỉ cần làm **một lần cho mỗi phiên** Colab.

In [ ]:
import os, os.path as osp, tarfile, time

TGZ_PATH  = '/content/drive/MyDrive/AI_Action/Data/v1.0-mini.tgz'  #@param {type:"string"}
DATAROOT  = '/content/nuscenes'                              #@param {type:"string"}

if not osp.exists('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

if osp.exists(osp.join(DATAROOT, 'v1.0-mini')):
    print('Da co du lieu tai', DATAROOT, '- bo qua buoc giai nen.')
else:
    assert osp.exists(TGZ_PATH), f'Khong thay file: {TGZ_PATH}. Hay sua lai TGZ_PATH.'
    os.makedirs(DATAROOT, exist_ok=True)
    t0 = time.time()
    print('Dang giai nen, vui long doi vai phut...')
    with tarfile.open(TGZ_PATH, 'r:gz') as tar:
        tar.extractall(DATAROOT)
    print(f'Xong sau {time.time() - t0:.0f} giay.')

print(sorted(os.listdir(DATAROOT)))

## Cell 3 — Nạp dữ liệu

Lệnh này đọc 13 bảng JSON trong `v1.0-mini/` vào bộ nhớ (khoảng 5–20 giây trên Colab).

In [ ]:
from nuscenes.nuscenes import NuScenes

nusc = NuScenes(version='v1.0-mini', dataroot=DATAROOT, verbose=True)

print('\n10 scene co trong ban mini:')
for s in nusc.scene:
    print(f"  {s['name']}  ({s['nbr_samples']:2d} keyframe)  {s['description']}")

## Cell 4 — Định nghĩa hàm vẽ

Chạy một lần, không in ra gì. Ba hàm chính:
- `get_keyframe()` — lấy point cloud và box của một keyframe
- `box_traces()` — chuyển mỗi box 3D thành khung dây 12 cạnh, cộng một đoạn chỉ hướng đầu xe
- `show_lidar_3d()` — dựng hình Plotly tương tác

In [ ]:
import numpy as np
import plotly.graph_objects as go
from nuscenes.utils.data_classes import LidarPointCloud

# Thu tu 8 goc cua devkit: 0-3 la mat truoc, 4-7 la mat sau.
EDGES = [(0, 1), (1, 2), (2, 3), (3, 0),
         (4, 5), (5, 6), (6, 7), (7, 4),
         (0, 4), (1, 5), (2, 6), (3, 7)]


def get_keyframe(scene_name, index, nsweeps=1):
    """Tra ve (point cloud Nx4, danh sach box, thong tin scene)."""
    scene = next(s for s in nusc.scene if s['name'] == scene_name)
    assert index < scene['nbr_samples'], \
        f"{scene_name} chi co {scene['nbr_samples']} keyframe (0..{scene['nbr_samples'] - 1})"
    token = scene['first_sample_token']
    for _ in range(index):
        token = nusc.get('sample', token)['next']
    sample = nusc.get('sample', token)
    lidar_token = sample['data']['LIDAR_TOP']

    # get_sample_data tu dong chuyen box ve he toa do cua lidar.
    path, boxes, _ = nusc.get_sample_data(lidar_token)
    if nsweeps > 1:
        pc, _ = LidarPointCloud.from_file_multisweep(
            nusc, sample, 'LIDAR_TOP', 'LIDAR_TOP', nsweeps=nsweeps)
    else:
        pc = LidarPointCloud.from_file(path)
    return pc.points.T, boxes, scene


def box_traces(boxes):
    """Gom box theo lop, moi lop mot trace de bam vao chu thich la an/hien duoc."""
    groups = {}
    for b in boxes:
        rgb = nusc.explorer.get_color(b.name)
        g = groups.setdefault(b.name, {'color': f'rgb{rgb}', 'x': [], 'y': [], 'z': [], 'n': 0})
        g['n'] += 1
        k = b.corners().T                      # (8, 3)
        for i, j in EDGES:                     # 12 canh
            g['x'] += [k[i, 0], k[j, 0], None]
            g['y'] += [k[i, 1], k[j, 1], None]
            g['z'] += [k[i, 2], k[j, 2], None]
        front = k[[0, 1, 2, 3]].mean(axis=0)   # doan chi huong dau xe
        g['x'] += [b.center[0], front[0], None]
        g['y'] += [b.center[1], front[1], None]
        g['z'] += [b.center[2], front[2], None]

    traces = []
    for name, g in sorted(groups.items(), key=lambda kv: -kv[1]['n']):
        traces.append(go.Scatter3d(
            x=g['x'], y=g['y'], z=g['z'], mode='lines',
            line=dict(color=g['color'], width=3),
            name=f"{name} ({g['n']})", hoverinfo='name'))
    return traces


def show_lidar_3d(scene_name='scene-0061', index=20, nsweeps=1,
                  color_by='height', max_points=60000, height=700, out_html=None):
    """Ve point cloud 3D tuong tac kem bounding box.

    color_by: 'height' (do cao z), 'intensity' (cuong do phan xa), 'distance' (khoang cach).
    """
    pts, boxes, scene = get_keyframe(scene_name, index, nsweeps)

    n_full = len(pts)
    if n_full > max_points:                    # giam diem cho nhe trinh duyet
        sel = np.random.default_rng(0).choice(n_full, max_points, replace=False)
        pts = pts[sel]

    if color_by == 'intensity':
        cval, cname = pts[:, 3], 'intensity'
    elif color_by == 'distance':
        cval, cname = np.linalg.norm(pts[:, :2], axis=1), 'khoang cach (m)'
    else:
        cval, cname = pts[:, 2], 'do cao z (m)'

    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode='markers',
        marker=dict(size=1.2, color=cval, colorscale='Viridis', opacity=0.85,
                    colorbar=dict(title=cname, thickness=12, len=0.6)),
        name=f'lidar ({len(pts):,} diem)', hoverinfo='skip'))

    for tr in box_traces(boxes):
        fig.add_trace(tr)

    fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[0], mode='markers',
                               marker=dict(size=6, color='red', symbol='x'),
                               name='xe (ego)'))

    title = (f"{scene['name']} · keyframe {index}/{scene['nbr_samples']} · "
             f"{len(boxes)} box · {n_full:,} diem ({nsweeps} sweep)")
    fig.update_layout(
        template='plotly_dark', height=height, title=title,
        margin=dict(l=0, r=0, t=40, b=0),
        legend=dict(itemsizing='constant', font=dict(size=10)),
        scene=dict(aspectmode='data',
                   xaxis_title='x (m)', yaxis_title='y (m)', zaxis_title='z (m)',
                   camera=dict(eye=dict(x=-1.4, y=-1.4, z=0.9))))

    if out_html:
        fig.write_html(out_html)
        print('Da luu', out_html)
    fig.show()
    return fig


print('Da dinh nghia xong cac ham.')

## Cell 5 — Hiển thị 3D tương tác

**Điều khiển bằng chuột:** kéo chuột trái = xoay · lăn chuột = zoom · kéo chuột phải = tịnh tiến · bấm vào tên lớp trong chú thích để ẩn/hiện lớp đó · nháy đúp để đặt lại góc nhìn.

Có thể đổi thông số ở thanh bên phải (Colab form) hoặc sửa trực tiếp trong code.

In [ ]:
SCENE    = 'scene-0796'  #@param ['scene-0061','scene-0103','scene-0553','scene-0655','scene-0757','scene-0796','scene-0916','scene-1077','scene-1094','scene-1100']
KEYFRAME = 23            #@param {type:"slider", min:0, max:38, step:1}
NSWEEPS  = 1             #@param {type:"slider", min:1, max:10, step:1}
COLOR_BY = 'height'      #@param ['height','intensity','distance']

fig = show_lidar_3d(SCENE, KEYFRAME, nsweeps=NSWEEPS, color_by=COLOR_BY)

## Cell 6 — Vài ví dụ khác

Bỏ dấu `#` ở dòng bạn muốn thử.

In [ ]:
# Canh ban dem, gop 5 sweep cho point cloud day hon:
show_lidar_3d('scene-1094', 10, nsweeps=5)

# To mau theo cuong do phan xa - vach ke duong va bien bao sang ro:
# show_lidar_3d('scene-0757', 20, color_by='intensity')

# Gop 10 sweep, lay toi da 150.000 diem:
# show_lidar_3d('scene-0061', 20, nsweeps=10, max_points=150000)

## Cell 7 (tuỳ chọn) — Đối chiếu với ảnh camera

Cùng các box 3D đó chiếu lên 6 ảnh camera, để thấy nhãn 3D và ảnh 2D dùng chung một bộ nhãn.

In [ ]:
import matplotlib.pyplot as plt
from nuscenes.utils.geometry_utils import BoxVisibility

CAM_LAYOUT = [['CAM_FRONT_LEFT', 'CAM_FRONT', 'CAM_FRONT_RIGHT'],
              ['CAM_BACK_LEFT',  'CAM_BACK',  'CAM_BACK_RIGHT']]

scene = next(s for s in nusc.scene if s['name'] == SCENE)
token = scene['first_sample_token']
for _ in range(KEYFRAME):
    token = nusc.get('sample', token)['next']
sample = nusc.get('sample', token)

fig2, axes = plt.subplots(2, 3, figsize=(22, 8))
for row, cams in enumerate(CAM_LAYOUT):
    for col, cam in enumerate(cams):
        cam_token = sample['data'][cam]
        _, bx, _ = nusc.get_sample_data(cam_token, box_vis_level=BoxVisibility.ANY)
        ax = axes[row][col]
        nusc.render_sample_data(cam_token, ax=ax, with_anns=True, verbose=False)
        ax.set_title(f'{cam}  ({len(bx)} box)', fontsize=11)
        ax.axis('off')
fig2.tight_layout()
plt.show()

## Cell 8 (tuỳ chọn) — Lưu thành file HTML và tải về

File HTML giữ nguyên khả năng xoay/zoom, mở được bằng trình duyệt bất kỳ mà không cần Python.

In [ ]:
show_lidar_3d(SCENE, KEYFRAME, nsweeps=NSWEEPS, out_html='lidar_3d.html')

from google.colab import files
files.download('lidar_3d.html')

---
## Xử lý sự cố

| Hiện tượng | Cách xử lý |
|---|---|
| `Khong thay file ... .tgz` | Sửa `TGZ_PATH` ở Cell 2 cho đúng đường dẫn trong Drive |
| Colab yêu cầu *RESTART SESSION* sau khi cài | Bấm nút đó, rồi chạy lại từ Cell 2 |
| Hình 3D không hiện | Chạy lại cell; nếu vẫn không được, giảm `max_points` xuống 30000 |
| Xoay bị giật | Giảm `max_points` (ví dụ 30000) hoặc `nsweeps=1` |
| `KEYFRAME` báo lỗi vượt giới hạn | Mỗi scene chỉ có 39–41 keyframe, chọn số nhỏ hơn |
| Hết RAM | Dùng `nsweeps=1`, và chạy `del fig` trước khi vẽ hình mới |

## Ghi chú

- Point cloud và box đều nằm trong **hệ toạ độ của lidar**: `x` hướng trước, `y` hướng trái, `z` hướng lên, gốc tại cảm biến lidar (cao 1,84 m so với mặt đất). Hàm `nusc.get_sample_data()` tự lo phần chuyển đổi này.
- Vùng trống quanh gốc toạ độ là **điểm mù**: thân xe che chính nó.
- Box chỉ tồn tại ở **keyframe** (2 Hz). Tăng `nsweeps` chỉ làm dày point cloud, số box không đổi.
- Vật đang chuyển động sẽ **bị nhoè** khi `nsweeps` lớn, vì phép bù chuyển động chỉ khử được chuyển động của xe mình.